<a href="https://colab.research.google.com/github/Aelton0/Aurora_Siger/blob/main/relatorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequencia de Lançamento da Aurora Siger

Importanto bibliotecas

In [3]:
import random

## Declaração das variaveis dos sensores
* Temperatura Interna = temp_int
* Temperatura Externa = temp_ext
* Integridade da Nave = integrity
* Voltagem dos Sistemas = energy_pct
* Pressão dos tanques em PSI = tank_pressure
* Status dos Modulos Criticos = critical_modules


Declaração das variavies de segurança

In [32]:
# Configurações de Segurança
SAFE_TEMP_INT = (15, 40)      # Entre 15°C e 40°C
SAFE_TEMP_EXT = (-150, 100)   # Entre -150°C e 100°C
MIN_ENERGY = 85               # Mínimo 85%
MIN_PRESSURE = 3000           # Mínimo 3000 PSI
MAX_PRESSURE = 4500           # Máximo 4500 PSI

# Função simulando a leitura dos sensores de telemetria
É criado um dicionario chamado 'dados', todo dicionario usa um par de chave:valor, a chave é a variavel e o valor é gerado pela biblioteca random.

Exemplo:
* temp_int : 'valor randomico'


Apesar de usar valores randomicos foi definido faixas.
* temp_int gera de 10 Cº até 50 Cº
* temp_ext gera de -180 Cº até 150 Cº
* energy_pct gera de 70% até 100%
* tank_pressure gera 2500 PSI até 5000 PSI
----
Para integrity e critical_modules foi aplicado uma logica de verificação nas faixas.

* Integrity é 1 se o valor gerado for maior que 0.1, logo tem 90% de chance de dar 1 (OK).

* critical_modules é 1 se o valor gerado for maior que  0.05, logo tem 95% de chance de dar 1(True)

OBS: o random.random pela documetação do Python gera valores de 0.0 até 1.0

random.random()

> Return the next random floating-point number in the range 0.0 <= X < 1.0






In [37]:
def realizar_telemetria():
    # Simulando a leitura dos sensores
    dados = {
        "temp_int": random.uniform(10, 50),
        "temp_ext": random.uniform(-180, 150),
        "integrity": 1 if random.random() > 0.1 else 0, # 90% de chance de estar ok
        "energy_pct": random.uniform(70, 100), # pct é de porcentagem
        "tank_pressure": random.uniform(2500, 5000),
        "critical_modules": True if random.random() > 0.05 else False
    }
    return dados


# Iniciando os calculos de autonomia
A formula é: energia_atual = capacidade_total_kwh * (carga_pct / 100)

* Consumo na decolagem 400kWh
* Perda de 50kWh

OBS: Como estou usando funções sera definido depois algumas variaveis como capacidade total, consumo na decolagem e perdas.

In [34]:
def calcular_autonomia(capacidade_total_kwh, carga_pct, consumo_decolagem, perdas):
    # Cálculo conforme a fórmula da atividade
    energia_atual = capacidade_total_kwh * (carga_pct / 100)
    autonomia_pos_decolagem = energia_atual - consumo_decolagem - perdas
    return autonomia_pos_decolagem

# Função para verificar os dados dos sensores
Nessa parte faço uma verificação de cada modulo:
* Temperatura
* Integridade
* Energia
* Pressão dos tanques
* Módulos Críticos (aqui é uma generalização por não foi dito oque considerar).

Aqui crio uma lista chamada 'erros' para armazenar a checagem dos modulos e caso de erro em algum armazena na lista para ser chamada depois.

Exemplo:
* if not executa um bloco de código se a condição for falsa, logo, se der SAFE_TEMP_INT = 0 então adiciona (append) "Temperatura Interna Crítica: 'dados da leitura telemetrica' °C" para dentro da lista 'erros'
* Outra maneira de ver isso é que é definido um intervalo, se a temp_int der um valor maior ou menor que o definido em SAFE_TEMP_INT então da 0 e adiciona a lista erro.

In [35]:
def verificar_decolagem(telemetria):
    erros = []

    # 1. Verificar Temperaturas
    if not (SAFE_TEMP_INT[0] <= telemetria["temp_int"] <= SAFE_TEMP_INT[1]):
        erros.append(f"Temperatura Interna Crítica: {telemetria['temp_int']:.2f}°C")

    # 2. Verificar Integridade (0 ou 1)
    if telemetria["integrity"] == 0:
        erros.append("Falha na Integridade Estrutural!")

    # 3. Verificar Energia
    if telemetria["energy_pct"] < MIN_ENERGY:
        erros.append(f"Energia Insuficiente: {telemetria['energy_pct']:.2f}%")

    # 4. Verificar Pressão
    if not (MIN_PRESSURE <= telemetria["tank_pressure"] <= MAX_PRESSURE):
        erros.append(f"Pressão dos Tanques Fora do Limite: {telemetria['tank_pressure']:.2f} PSI")

    # 5. Módulos Críticos
    if not telemetria["critical_modules"]:
        erros.append("Erro em Módulos Críticos do Sistema")

    return erros

# Inicialização do lançamento

In [38]:
# --- INICIANDO ---
telemetria_atual = realizar_telemetria()  #Chama a função realizar_telemetria e armazena na variavel telemetria_atual
falhas = verificar_decolagem(telemetria_atual) #Chama a função verificar_decolagem e armazena em falhas

print("=== RELATÓRIO DE TELEMETRIA ===")
for chave, valor in telemetria_atual.items():
    print(f"{chave}: {valor:.2f}" if isinstance(valor, float) else f"{chave}: {valor}")

print("\n=== ANÁLISE ENERGÉTICA ===")
# Só para lembrar a ordem dos argumentos da função calcular_autonomia() (capacidade_total_kwh, carga_pct, consumo_decolagem, perdas)
# Exemplo: 5000kWh total, consumo de 400kWh na subida, perda de 50kWh
autonomia = calcular_autonomia(5000, telemetria_atual["energy_pct"], 400, 50)
print(f"Autonomia estimada após decolagem: {autonomia:.2f} kWh")

print("\n" + "="*30)
if not falhas:
    print("STATUS: PRONTO PARA DECOLAR")
else:
    print("STATUS: DECOLAGEM ABORTADA")
    print("MOTIVOS:")
    for erro in falhas:
        print(f"- {erro}")

=== RELATÓRIO DE TELEMETRIA ===
temp_int: 39.32
temp_ext: 16.76
integrity: 1
energy_pct: 94.22
tank_pressure: 4286.87
critical_modules: True

=== ANÁLISE ENERGÉTICA ===
Autonomia estimada após decolagem: 4260.94 kWh

STATUS: PRONTO PARA DECOLAR
